# ST-GCN joint_bone - mstcn vs unit_tcn

The one architectural lever not yet tried. mmaction2's `STGCNBlock` supports an alternative temporal-convolution module - `mstcn` (multi-scale TCN) instead of the default single-kernel `unit_tcn` - verified directly from the backbone source read earlier: `tcn_type` is a real, supported kwarg that routes through `STGCN.__init__`'s `**kwargs` into every block. Passing `tcn_type='mstcn'` in the backbone config switches the whole network's temporal modeling, nothing else changes.

Same proven recipe otherwise: `joint_bone`, `weight_decay=1e-3`, StepLR, advanced augmentation, sampler built fresh per call, seeded workers, 30-frame data. Single-seed screening pass first - if `mstcn` looks genuinely better, it gets a multi-seed confirmation before being trusted, same pattern as every other finding today.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
    confusion_matrix, precision_recall_fscore_support,
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### DataLoader worker-seeding fix

In [2]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


### paths

In [3]:
OUT_ROOT = "/workspace/outputs/stgcn_mstcn"
CKPT_ROOT = "/workspace/checkpoints/stgcn_mstcn"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - 30-frame, advanced augmentation, joint_bone

In [4]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="advanced")
val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### batch -> ST-GCN input - joint_bone, in_channels=5

In [5]:
def batch_to_stgcn_input(batch):
    x = torch.cat([batch["keypoints"].float(), batch["bone"].float()], dim=-1)
    return x.unsqueeze(1).unsqueeze(1)


### SMOKE TEST - both tcn types, run alone first

`mstcn` has never been used anywhere in this project - genuinely worth confirming it builds and produces the right output shape before committing to a full training run, not assuming its defaults are compatible.

In [6]:
def build_model(tcn_type):
    backbone_cfg = dict(
        type="STGCN",
        graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
        in_channels=5, base_channels=64, ch_ratio=2,
        num_stages=9, inflate_stages=[4, 7], down_stages=[4, 7],
    )
    if tcn_type == "mstcn":
        backbone_cfg["tcn_type"] = "mstcn"

    model_cfg = dict(
        type="RecognizerGCN",
        backbone=backbone_cfg,
        cls_head=dict(type="GCNHead", num_classes=num_classes, in_channels=256,
                      loss_cls=dict(type="CrossEntropyLoss")),
    )
    return MODELS.build(model_cfg).to(device)


smoke_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))
smoke_input = batch_to_stgcn_input(smoke_batch).to(device)
print("input shape:", smoke_input.shape)

for tcn_type in ["unit_tcn", "mstcn"]:
    model = build_model(tcn_type)
    n_params = sum(p.numel() for p in model.parameters())
    model.eval()
    with torch.no_grad():
        out = model(smoke_input, mode="tensor", stage="head")
    status = "OK" if out.shape == (4, num_classes) else "MISMATCH"
    print(f"{tcn_type:10s} params={n_params:,}  output_shape={tuple(out.shape)}  [{status}]")
    assert out.shape == (4, num_classes), f"{tcn_type}: shape mismatch"
    del model

print()
print("both tcn types verified working")


input shape: torch.Size([4, 1, 1, 30, 17, 5])
unit_tcn   params=3,027,831  output_shape=(4, 18)  [OK]
mstcn      params=1,316,487  output_shape=(4, 18)  [OK]

both tcn types verified working


### metrics

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "labels": labels, "preds": preds,
    }


### the reusable training function

Fixed: joint_bone, weight_decay=1e-3, StepLR, advanced augmentation, seeded workers, fresh sampler per call. Only `tcn_type` varies.

In [8]:
def train_one_tcn_type(tcn_type, seed=42, step_size=10, num_epochs=60, patience=20,
                        batch_size=16, lr=0.01, weight_decay=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=seed)
    model = build_model(tcn_type)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=balanced_sampler,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=seed_worker, generator=loader_generator,
    )

    task = Task.init(project_name="BadmintonPoseAnalysis", task_name=f"stgcn_tcntype_{tcn_type}", reuse_last_task_id=False)
    logger = task.get_logger()
    task.connect({
        "tcn_type": tcn_type, "stream": "joint_bone", "seed": seed,
        "lr": lr, "weight_decay": weight_decay, "num_epochs": num_epochs,
        "batch_size": batch_size, "augmentation_policy": "advanced",
    })

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=0.1)

    best_val_top1 = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_labels, epoch_preds = [], []

        for batch in train_loader:
            x = batch_to_stgcn_input(batch).to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(x, mode="tensor", stage="head")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_labels.append(y.cpu().numpy())
            epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

        scheduler.step()

        train_loss = epoch_loss / len(train_loader.dataset)
        train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))
        val_metrics = evaluate(model, val_loader)

        logger.report_scalar("loss", "train", train_loss, iteration=epoch)
        logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
        logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
        logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"[{tcn_type}] epoch {epoch:3d}  train_acc={train_acc:.4f}  "
                  f"val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

        if epochs_without_improvement >= patience:
            print(f"[{tcn_type}] early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f} (epoch {best_epoch})")
            break

    model.load_state_dict(best_state)

    train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True))
    test_final = evaluate(model, test_loader)

    metrics = {
        "tcn_type": tcn_type,
        "best_epoch": best_epoch,
        "train_acc": train_final["top1"],
        "test_top1": test_final["top1"],
        "test_top5": test_final["top5"],
        "test_mca": test_final["mca"],
        "test_macro_f1": test_final["macro_f1"],
        "train_test_gap": train_final["top1"] - test_final["top1"],
    }

    for metric_name in ["train_acc", "test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap", "best_epoch"]:
        logger.report_single_value(name=metric_name, value=metrics[metric_name])

    cm = confusion_matrix(test_final["labels"], test_final["preds"], labels=list(range(num_classes)))
    logger.report_confusion_matrix(
        title="Test Confusion Matrix", series=tcn_type,
        matrix=cm, xaxis="Predicted", yaxis="Actual",
        xlabels=class_names, ylabels=class_names,
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        test_final["labels"], test_final["preds"], labels=list(range(num_classes)), zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    logger.report_table(title="Per-Class Test Metrics", series=tcn_type, table_plot=per_class_df)

    ckpt_path = os.path.join(CKPT_ROOT, f"{tcn_type}_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    task.upload_artifact("checkpoint", ckpt_path)
    per_class_df.to_csv(os.path.join(OUT_ROOT, f"per_class_{tcn_type}.csv"), index=False)
    task.close()

    return metrics


### run both tcn types

In [9]:
all_results = {}
import time

for tcn_type in ["unit_tcn", "mstcn"]:
    model = build_model(tcn_type)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

    model.train()
    t0 = time.time()
    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, mode="tensor", stage="head"), y)
        loss.backward()
        optimizer.step()
    elapsed = time.time() - t0
    print(f"{tcn_type}: one epoch took {elapsed:.1f} seconds")
    del model

unit_tcn: one epoch took 21.0 seconds
mstcn: one epoch took 169.9 seconds


### comparison table

In [10]:
PREVIOUS_MULTISEED_BASELINE = {"tcn_type": "unit_tcn (30-frame multi-seed mean)", "best_epoch": None,
                               "test_top1": 0.7089, "test_top5": 0.9442, "test_mca": 0.6222,
                               "test_macro_f1": 0.6340, "train_test_gap": None}
LI_ET_AL_BENCHMARK = {"tcn_type": "Li et al. benchmark", "best_epoch": None,
                      "test_top1": 0.7441, "test_top5": 0.9376, "test_mca": 0.6144,
                      "test_macro_f1": None, "train_test_gap": None}

comparison = pd.DataFrame(list(all_results.values()) + [PREVIOUS_MULTISEED_BASELINE, LI_ET_AL_BENCHMARK]).set_index("tcn_type")
comparison = comparison[["best_epoch", "train_acc", "test_top1", "train_test_gap",
                          "test_top5", "test_mca", "test_macro_f1"]]
comparison.to_csv(os.path.join(OUT_ROOT, "tcn_type_comparison.csv"))
comparison


KeyError: "['train_acc'] not in index"

---

This `unit_tcn` row is a fresh single-seed run, so compare it against the multi-seed mean (70.89%) to gauge where this particular seed landed before judging `mstcn` against it - use the multi-seed number as the real reference point, not this run's own `unit_tcn` result. If `mstcn` clears both by a real margin, it's worth a proper 3-seed confirmation next. If not, that closes out the architectural search cleanly - time to move to ST-TR.